In [1]:
!pip install dagshub mlflow imbalanced-learn xgboost --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 695.9 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 35.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [2]:
import gc
import time
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import dagshub
import mlflow
import mlflow.sklearn

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

data_path = "/kaggle/input/competitions/ieee-fraud-detection"
seed = 42

from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)

In [3]:
def find_best_f1_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)

    best_index = np.argmax(f1_scores[:-1])
    best_threshold = thresholds[best_index]
    best_f1 = f1_scores[best_index]

    return best_threshold, best_f1


def calculate_classification_metrics(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def log_metrics(prefix, metrics):
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(f"{prefix}_{metric_name}", metric_value)

In [4]:
dagshub.init(
    repo_owner="lmars23",
    repo_name="ml_assignment_2",
    mlflow=True
)

mlflow.set_experiment("XGBoost_Training")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f07a5b60-5c24-4ab5-9e63-c6b4b343f3d7&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=73e29540b4e79ba10daf42dd761440a60a6ce5a71d8c456808520e93f794a946




Accessing as lmars23

Initialized MLflow to track repo "lmars23/ml_assignment_2"

Repository lmars23/ml_assignment_2 initialized!

<Experiment: artifact_location='mlflow-artifacts:/b24b7c5473ff46b1a69c3076e95fe45a', creation_time=1777841740558, experiment_id='3', last_update_time=1777841740558, lifecycle_stage='active', name='XGBoost_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## Helper functions

In [5]:
def reduce_memory_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "object":
            continue

        c_min = df[col].min()
        c_max = df[col].max()

        if pd.isna(c_min) or pd.isna(c_max):
            continue

        if str(col_type).startswith("int"):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        else:
            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df


def show_auc(y_train_true, train_pred, y_val_true, val_pred):
    train_auc = roc_auc_score(y_train_true, train_pred)
    val_auc = roc_auc_score(y_val_true, val_pred)
    return train_auc, val_auc, train_auc - val_auc


# Feature Engineering and Selection

In [6]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()

        if "TransactionDT" in X.columns:
            X["Transaction_hour"] = (X["TransactionDT"] // 3600) % 24

        return X


class VColumnReducer(BaseEstimator, TransformerMixin):
    def __init__(self, use_reduction=True, corr_threshold=0.75):
        self.use_reduction = use_reduction
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        self.original_columns_ = X.columns.tolist()
        v_cols = [c for c in X.columns if c.startswith("V") and c[1:].isdigit()]

        if not self.use_reduction:
            self.selected_v_columns_ = v_cols
            self.dropped_v_columns_ = []
            self.keep_columns_ = self.original_columns_
            return self

        groups = {}
        for col in v_cols:
            groups.setdefault(int(X[col].isna().sum()), []).append(col)

        selected = []
        dropped = []

        for _, cols in groups.items():
            if len(cols) == 1:
                selected.append(cols[0])
                continue

            corr = X[cols].corr().abs()
            visited = set()

            for col in cols:
                if col in visited:
                    continue

                stack = [col]
                component = []
                visited.add(col)

                while stack:
                    current = stack.pop()
                    component.append(current)
                    neighbours = corr.index[corr.loc[current] > self.corr_threshold].tolist()

                    for nb in neighbours:
                        if nb not in visited:
                            visited.add(nb)
                            stack.append(nb)

                best_col = max(component, key=lambda c: X[c].nunique(dropna=True))
                selected.append(best_col)
                dropped.extend([c for c in component if c != best_col])

        self.selected_v_columns_ = selected
        self.dropped_v_columns_ = dropped

        non_v_cols = [c for c in self.original_columns_ if c not in v_cols]
        keep_set = set(non_v_cols + selected)
        self.keep_columns_ = [c for c in self.original_columns_ if c in keep_set]
        return self

    def transform(self, X):
        return X[[c for c in self.keep_columns_ if c in X.columns]].copy()


class MissingValueDropper(BaseEstimator, TransformerMixin):
    def __init__(self, missing_limit=0.90):
        self.missing_limit = missing_limit

    def fit(self, X, y=None):
        missing_rate = X.isna().mean()
        self.keep_columns_ = missing_rate[missing_rate <= self.missing_limit].index.tolist()
        self.dropped_columns_ = missing_rate[missing_rate > self.missing_limit].index.tolist()
        return self

    def transform(self, X):
        return X[[c for c in self.keep_columns_ if c in X.columns]].copy()


class IVFilter(BaseEstimator, TransformerMixin):
    def __init__(self, use_filter=True, iv_limit=0.02, bins=10):
        self.use_filter = use_filter
        self.iv_limit = iv_limit
        self.bins = bins

    def _feature_iv(self, s, y):
        s = pd.Series(s).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)

        try:
            if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > self.bins:
                x = pd.qcut(s, q=self.bins, duplicates="drop")
                x = x.astype("object").where(~x.isna(), "missing")
            else:
                x = s.astype("object").where(~s.isna(), "missing")

            tmp = pd.DataFrame({"x": x, "y": y})
            grouped = tmp.groupby("x", observed=False)["y"].agg(["count", "sum"])

            bad = grouped["sum"] + 0.5
            good = grouped["count"] - grouped["sum"] + 0.5

            dist_bad = bad / bad.sum()
            dist_good = good / good.sum()

            iv = ((dist_bad - dist_good) * np.log(dist_bad / dist_good)).sum()

            if pd.isna(iv) or np.isinf(iv):
                return 0

            return float(iv)
        except Exception:
            return 0

    def fit(self, X, y):
        self.all_columns_ = X.columns.tolist()

        if not self.use_filter:
            self.iv_values_ = pd.DataFrame({"feature": self.all_columns_, "iv": np.nan})
            self.selected_features_ = self.all_columns_
            self.dropped_features_ = []
            return self

        values = []
        for col in X.columns:
            values.append({"feature": col, "iv": self._feature_iv(X[col], y)})

        self.iv_values_ = pd.DataFrame(values).sort_values("iv", ascending=False).reset_index(drop=True)
        self.selected_features_ = self.iv_values_[self.iv_values_["iv"] >= self.iv_limit]["feature"].tolist()

        if len(self.selected_features_) == 0:
            self.selected_features_ = self.all_columns_

        self.dropped_features_ = [c for c in self.all_columns_ if c not in self.selected_features_]
        return self

    def transform(self, X):
        return X[[c for c in self.selected_features_ if c in X.columns]].copy()


## Load and merge data

In [7]:
train_transaction = pd.read_csv(data_path + "/train_transaction.csv")
train_identity = pd.read_csv(data_path + "/train_identity.csv")
test_transaction = pd.read_csv(data_path + "/test_transaction.csv")
test_identity = pd.read_csv(data_path + "/test_identity.csv")
sample_submission = pd.read_csv(data_path + "/sample_submission.csv")

test_identity.columns = test_identity.columns.str.replace("-", "_")

train_transaction = reduce_memory_usage(train_transaction)
train_identity = reduce_memory_usage(train_identity)
test_transaction = reduce_memory_usage(test_transaction)
test_identity = reduce_memory_usage(test_identity)

train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

print("train shape:", train.shape)
print("test shape:", test.shape)

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()


Memory: 2062.07 MB -> 1203.22 MB
Memory: 143.14 MB -> 129.94 MB
Memory: 1771.84 MB -> 1038.31 MB
Memory: 140.08 MB -> 127.09 MB
train shape: (590540, 434)
test shape: (506691, 433)


150

## Time based validation split

In [8]:
train = train.sort_values("TransactionDT").reset_index(drop=True)

split_at = int(len(train) * 0.8)
train_part = train.iloc[:split_at].copy()
val_part = train.iloc[split_at:].copy()

X_train = train_part.drop(["isFraud", "TransactionID"], axis=1)
y_train = train_part["isFraud"]

X_val = val_part.drop(["isFraud", "TransactionID"], axis=1)
y_val = val_part["isFraud"]

X_full = train.drop(["isFraud", "TransactionID"], axis=1)
y_full = train["isFraud"]

X_test = test.drop(["TransactionID"], axis=1)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("train fraud rate:", round(y_train.mean(), 5))
print("val fraud rate:", round(y_val.mean(), 5))


X_train: (472432, 432)
X_val: (118108, 432)
X_full: (590540, 432)
X_test: (506691, 432)
train fraud rate: 0.03514
val fraud rate: 0.03441


## Experiment switches

In [9]:
use_smote_values = [False]

use_v_reduction = True
v_corr_threshold = 0.80

missing_limit = 0.95

use_iv_filter = False
iv_limit = 0.01
iv_bins = 10

# Preprocessing

In [10]:
def make_preprocessor():
    numeric_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    cat_pipe = SkPipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.float32))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, make_column_selector(dtype_include=np.number)),
        ("cat", cat_pipe, make_column_selector(dtype_include=object))
    ], remainder="drop")


def make_feature_steps():
    return [
        ("feature_engineering", FeatureEngineer()),
        ("v_reducer", VColumnReducer(use_reduction=use_v_reduction, corr_threshold=v_corr_threshold)),
        ("missing_dropper", MissingValueDropper(missing_limit=missing_limit)),
        ("iv_filter", IVFilter(use_filter=use_iv_filter, iv_limit=iv_limit, bins=iv_bins)),
        ("preprocessing", make_preprocessor()),
        ("variance_filter", VarianceThreshold(threshold=0.0)),
    ]


## Fit data preparation 

In [11]:
feature_pipe = SkPipeline(make_feature_steps())

print("Fitting feature engineering / V reduction / IV filtering / preprocessing once...")
start = time.time()

X_train_ready = feature_pipe.fit_transform(X_train, y_train)
X_val_ready = feature_pipe.transform(X_val)

prepare_seconds = time.time() - start

print("prepare seconds:", round(prepare_seconds, 2))
print("prepared train:", X_train_ready.shape)
print("prepared val:", X_val_ready.shape)
print("V columns kept:", len(feature_pipe.named_steps["v_reducer"].selected_v_columns_))
print("V columns dropped:", len(feature_pipe.named_steps["v_reducer"].dropped_v_columns_))
print("IV features kept:", len(feature_pipe.named_steps["iv_filter"].selected_features_))
print("IV features dropped:", len(feature_pipe.named_steps["iv_filter"].dropped_features_))


Fitting feature engineering / V reduction / IV filtering / preprocessing once...
prepare seconds: 41.02
prepared train: (472432, 220)
prepared val: (118108, 220)
V columns kept: 134
V columns dropped: 205
IV features kept: 220
IV features dropped: 0


# Training

In [12]:
model_settings_list = [
    {
        "n_estimators": 600,
        "learning_rate": 0.035,
        "max_depth": 8,
        "min_child_weight": 20,
        "subsample": 0.70,
        "colsample_bytree": 0.60,
        "reg_lambda": 10.0,
        "reg_alpha": 1.0
    },

    # {
    #     "n_estimators": 700,
    #     "learning_rate": 0.04,
    #     "max_depth": 7,
    #     "min_child_weight": 10,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

    
    # {
    #     "n_estimators": 800,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 10,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 5.0,
    #     "reg_alpha": 0.1
    # },

    
    # {
    #     "n_estimators": 800,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 8,
    #     "subsample": 0.85,
    #     "colsample_bytree": 0.60,
    #     "reg_lambda": 4.0,
    #     "reg_alpha": 0.2
    # },

    
    # {
    #     "n_estimators": 900,
    #     "learning_rate": 0.03,
    #     "max_depth": 5,
    #     "min_child_weight": 20,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.70,
    #     "reg_lambda": 8.0,
    #     "reg_alpha": 1.0
    # },

   
    # {
    #     "n_estimators": 700,
    #     "learning_rate": 0.04,
    #     "max_depth": 7,
    #     "min_child_weight": 15,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 8.0,
    #     "reg_alpha": 0.7
    # },

   
    # {
    #     "n_estimators": 700,
    #     "learning_rate": 0.04,
    #     "max_depth": 7,
    #     "min_child_weight": 10,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.60,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

   
    # {
    #     "n_estimators": 750,
    #     "learning_rate": 0.04,
    #     "max_depth": 6,
    #     "min_child_weight": 10,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

    
    # {
    #     "n_estimators": 1000,
    #     "learning_rate": 0.025,
    #     "max_depth": 7,
    #     "min_child_weight": 10,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

    
    # {
    #     "n_estimators": 900,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 10,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.2
    # },

    
    # {
    #     "n_estimators": 800,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 15,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 8.0,
    #     "reg_alpha": 0.5
    # },

    
    # {
    #     "n_estimators": 850,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 10,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.65,
    #     "reg_lambda": 5.0,
    #     "reg_alpha": 0.2
    # },

    
    # {
    #     "n_estimators": 850,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 8,
    #     "subsample": 0.85,
    #     "colsample_bytree": 0.65,
    #     "reg_lambda": 4.0,
    #     "reg_alpha": 0.2
    # },

    
    # {
    #     "n_estimators": 850,
    #     "learning_rate": 0.03,
    #     "max_depth": 6,
    #     "min_child_weight": 12,
    #     "subsample": 0.85,
    #     "colsample_bytree": 0.60,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

   
    # {
    #     "n_estimators": 1100,
    #     "learning_rate": 0.025,
    #     "max_depth": 5,
    #     "min_child_weight": 15,
    #     "subsample": 0.85,
    #     "colsample_bytree": 0.75,
    #     "reg_lambda": 6.0,
    #     "reg_alpha": 0.5
    # },

   
    # {
    #     "n_estimators": 1000,
    #     "learning_rate": 0.03,
    #     "max_depth": 5,
    #     "min_child_weight": 25,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.70,
    #     "reg_lambda": 10.0,
    #     "reg_alpha": 1.0
    # },


   
    # {
    #     "n_estimators": 600,
    #     "learning_rate": 0.05,
    #     "max_depth": 7,
    #     "min_child_weight": 15,
    #     "subsample": 0.75,
    #     "colsample_bytree": 0.70,
    #     "reg_lambda": 8.0,
    #     "reg_alpha": 0.7
    # },

   
    # {
    #     "n_estimators": 1200,
    #     "learning_rate": 0.02,
    #     "max_depth": 6,
    #     "min_child_weight": 12,
    #     "subsample": 0.80,
    #     "colsample_bytree": 0.70,
    #     "reg_lambda": 7.0,
    #     "reg_alpha": 0.5
    # }
]

In [13]:
positive_count = int((y_train == 1).sum())
negative_count = int((y_train == 0).sum())
scale_pos_weight = negative_count / positive_count
print("scale_pos_weight:", round(scale_pos_weight, 4))

def make_model(settings):
    return XGBClassifier(
        n_estimators=settings["n_estimators"],
        learning_rate=settings["learning_rate"],
        max_depth=settings["max_depth"],
        min_child_weight=settings["min_child_weight"],
        subsample=settings["subsample"],
        colsample_bytree=settings["colsample_bytree"],
        reg_lambda=settings["reg_lambda"],
        reg_alpha=settings["reg_alpha"],
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )


def make_full_pipeline(settings, use_smote):
    steps = make_feature_steps()
    if use_smote:
        steps.append(("smote", SMOTE(random_state=seed, k_neighbors=5)))
    steps.append(("model", make_model(settings)))
    return ImbPipeline(steps)

results = []
best_val_auc = -1
best_settings = None
best_use_smote = None
best_run_name = None

with mlflow.start_run(run_name="xgboost_experiment"):
    mlflow.log_param("pipeline", "FeatureEngineer -> VColumnReducer -> MissingValueDropper -> IVFilter -> Preprocessor -> VarianceThreshold -> optional SMOTE -> XGBoost")
    mlflow.log_param("validation_type", "time_based_80_20")
    mlflow.log_param("missing_limit", missing_limit)
    mlflow.log_param("use_v_reduction", use_v_reduction)
    mlflow.log_param("v_corr_threshold", v_corr_threshold)
    mlflow.log_param("use_iv_filter", use_iv_filter)
    mlflow.log_param("iv_limit", iv_limit)
    mlflow.log_param("iv_bins", iv_bins)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_metric("prepare_seconds", prepare_seconds)
    mlflow.log_metric("prepared_feature_count", X_train_ready.shape[1])

    for use_smote in use_smote_values:
        if use_smote:
            print("Applying SMOTE once for this group of XGBoost runs...")
            X_fit, y_fit = SMOTE(random_state=seed, k_neighbors=5).fit_resample(X_train_ready, y_train)
        else:
            X_fit, y_fit = X_train_ready, y_train

        for settings in model_settings_list:
            run_name = f"XGB_trees_{settings['n_estimators']}_depth_{settings['max_depth']}_smote_{use_smote}"
            print("=" * 80)
            print(run_name)

            with mlflow.start_run(run_name=run_name, nested=True):
                mlflow.log_param("model_type", "XGBClassifier")
                mlflow.log_param("use_smote", use_smote)
                for key, value in settings.items():
                    mlflow.log_param(key, value)

                model = make_model(settings)
                start = time.time()
                model.fit(X_fit, y_fit)
                fit_seconds = time.time() - start

                train_pred = model.predict_proba(X_train_ready)[:, 1]
                val_pred = model.predict_proba(X_val_ready)[:, 1]
                
                best_threshold, best_threshold_f1 = find_best_f1_threshold(y_val, val_pred)
                
                train_metrics_05 = calculate_classification_metrics(y_train, train_pred, 0.5)
                val_metrics_05 = calculate_classification_metrics(y_val, val_pred, 0.5)
                
                train_metrics_best = calculate_classification_metrics(y_train, train_pred, best_threshold)
                val_metrics_best = calculate_classification_metrics(y_val, val_pred, best_threshold)
                
                train_auc = train_metrics_05["roc_auc"]
                val_auc = val_metrics_05["roc_auc"]
                auc_gap = train_auc - val_auc
                
                average_precision_gap = train_metrics_05["average_precision"] - val_metrics_05["average_precision"]
                f1_gap_best_threshold = train_metrics_best["f1"] - val_metrics_best["f1"]
                
                mlflow.log_metric("fit_seconds", fit_seconds)
                mlflow.log_metric("train_auc", train_auc)
                mlflow.log_metric("val_auc", val_auc)
                mlflow.log_metric("auc_gap", auc_gap)
                
                mlflow.log_metric("best_f1_threshold", best_threshold)
                mlflow.log_metric("best_threshold_f1", best_threshold_f1)
                mlflow.log_metric("average_precision_gap", average_precision_gap)
                mlflow.log_metric("f1_gap_best_threshold", f1_gap_best_threshold)
                
                log_metrics("train_t05", train_metrics_05)
                log_metrics("val_t05", val_metrics_05)
                
                log_metrics("train_best_threshold", train_metrics_best)
                log_metrics("val_best_threshold", val_metrics_best)

                results.append({
                    "run_name": run_name,
                    "use_smote": use_smote,
                    "fit_seconds": fit_seconds,
                
                    "train_auc": train_auc,
                    "val_auc": val_auc,
                    "auc_gap": auc_gap,
                
                    "train_average_precision": train_metrics_05["average_precision"],
                    "val_average_precision": val_metrics_05["average_precision"],
                    "average_precision_gap": average_precision_gap,
                
                    "val_accuracy_t05": val_metrics_05["accuracy"],
                    "val_balanced_accuracy_t05": val_metrics_05["balanced_accuracy"],
                    "val_precision_t05": val_metrics_05["precision"],
                    "val_recall_t05": val_metrics_05["recall"],
                    "val_f1_t05": val_metrics_05["f1"],
                
                    "best_f1_threshold": best_threshold,
                    "best_threshold_f1": best_threshold_f1,
                    "val_precision_best_threshold": val_metrics_best["precision"],
                    "val_recall_best_threshold": val_metrics_best["recall"],
                    "val_f1_best_threshold": val_metrics_best["f1"],
                    "f1_gap_best_threshold": f1_gap_best_threshold,
                
                    "val_tn": val_metrics_best["tn"],
                    "val_fp": val_metrics_best["fp"],
                    "val_fn": val_metrics_best["fn"],
                    "val_tp": val_metrics_best["tp"],
                
                    **settings
                })

                print("train_auc:", round(train_auc, 6))
                print("val_auc:", round(val_auc, 6))
                print("auc_gap:", round(auc_gap, 6))
                print("fit seconds:", round(fit_seconds, 2))
                print("val_average_precision:", round(val_metrics_05["average_precision"], 6))
                print("val_precision_t05:", round(val_metrics_05["precision"], 6))
                print("val_recall_t05:", round(val_metrics_05["recall"], 6))
                print("val_f1_t05:", round(val_metrics_05["f1"], 6))
                print("best_f1_threshold:", round(best_threshold, 6))
                print("val_f1_best_threshold:", round(val_metrics_best["f1"], 6))
                
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_settings = settings.copy()
                    best_use_smote = use_smote
                    best_run_name = run_name

                gc.collect()

    results_df = pd.DataFrame(results).sort_values("val_auc", ascending=False)
    results_df.to_csv("xgboost_results.csv", index=False)
    mlflow.log_artifact("xgboost_results.csv")

    print("=" * 80)
    print("Best validation AUC:", best_val_auc)
    print("Best run:", best_run_name)
    display(results_df)

    print("Fitting final best pipeline on full train for inference...")
    final_start = time.time()
    best_pipeline = make_full_pipeline(best_settings, best_use_smote)
    best_pipeline.fit(X_full, y_full)
    final_fit_seconds = time.time() - final_start

    test_pred = best_pipeline.predict_proba(X_test)[:, 1]
    submission = sample_submission.copy()
    submission["isFraud"] = test_pred
    submission.to_csv("submission_xgboost.csv", index=False)

    with mlflow.start_run(run_name="best_xgboost_model", nested=True):
        mlflow.log_param("best_run_name", best_run_name)
        mlflow.log_param("use_smote", best_use_smote)
        mlflow.log_param("pipeline", "full raw-data inference pipeline")
        mlflow.log_param("scale_pos_weight", scale_pos_weight)
        for key, value in best_settings.items():
            mlflow.log_param(key, value)
        mlflow.log_metric("best_validation_auc", best_val_auc)
        mlflow.log_metric("final_fit_seconds", final_fit_seconds)

print("Saved submission_xgboost.csv")


scale_pos_weight: 27.4615
XGB_trees_600_depth_8_smote_False
train_auc: 0.986185
val_auc: 0.917421
auc_gap: 0.068764
fit seconds: 80.27
val_average_precision: 0.536411
val_precision_t05: 0.291088
val_recall_t05: 0.69685
val_f1_t05: 0.410643
best_f1_threshold: 0.735155
val_f1_best_threshold: 0.51687
🏃 View run XGB_trees_600_depth_8_smote_False at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/c97e886963ee40deb2d22790ef353243
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
Best validation AUC: 0.9174214753882118
Best run: XGB_trees_600_depth_8_smote_False


,run_name,use_smote,fit_seconds,train_auc,val_auc,auc_gap,train_average_precision,val_average_precision,average_precision_gap,val_accuracy_t05,val_balanced_accuracy_t05,val_precision_t05,val_recall_t05,val_f1_t05,best_f1_threshold,best_threshold_f1,val_precision_best_threshold,val_recall_best_threshold,val_f1_best_threshold,f1_gap_best_threshold,val_tn,val_fp,val_fn,val_tp,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha
0,XGB_trees_600_depth_8_smote_False,False,80.267376,0.986185,0.917421,0.068764,0.84331,0.536411,0.306898,0.931173,0.818187,0.291088,0.69685,0.410643,0.735155,0.51687,0.508205,0.525837,0.51687,0.236812,111976,2068,1927,2137,600,0.035,8,20,0.7,0.6,10.0,1.0


Fitting final best pipeline on full train for inference...
🏃 View run best_xgboost_model at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/569ce60ffeb4472b904c660d7eda2cd7
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
🏃 View run xgboost_experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/33dd7cd7de3648bbb8053196714ce2af
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
Saved submission_xgboost.csv


In [14]:
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, TransformerMixin

class ColumnIndexSelector(BaseEstimator, TransformerMixin):
    def __init__(self, selected_indices):
        self.selected_indices = selected_indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[:, self.selected_indices]


def make_full_pipeline_with_column_selector(settings, use_smote, selected_indices):
    steps = make_feature_steps()

    steps.append((
        "permutation_column_selector",
        ColumnIndexSelector(selected_indices=selected_indices)
    ))

    if use_smote:
        steps.append(("smote", SMOTE(random_state=seed, k_neighbors=5)))

    steps.append(("model", make_model(settings)))

    return ImbPipeline(steps)


def calculate_auc_metrics_for_selected_model(model, X_train_part, y_train_part, X_val_part, y_val_part):
    train_pred = model.predict_proba(X_train_part)[:, 1]
    val_pred = model.predict_proba(X_val_part)[:, 1]

    best_threshold, best_threshold_f1 = find_best_f1_threshold(y_val_part, val_pred)

    train_metrics_05 = calculate_classification_metrics(y_train_part, train_pred, 0.5)
    val_metrics_05 = calculate_classification_metrics(y_val_part, val_pred, 0.5)

    train_metrics_best = calculate_classification_metrics(y_train_part, train_pred, best_threshold)
    val_metrics_best = calculate_classification_metrics(y_val_part, val_pred, best_threshold)

    return {
        "train_pred": train_pred,
        "val_pred": val_pred,
        "best_threshold": best_threshold,
        "best_threshold_f1": best_threshold_f1,
        "train_metrics_05": train_metrics_05,
        "val_metrics_05": val_metrics_05,
        "train_metrics_best": train_metrics_best,
        "val_metrics_best": val_metrics_best,
        "train_auc": train_metrics_05["roc_auc"],
        "val_auc": val_metrics_05["roc_auc"],
        "auc_gap": train_metrics_05["roc_auc"] - val_metrics_05["roc_auc"],
        "average_precision_gap": train_metrics_05["average_precision"] - val_metrics_05["average_precision"],
        "f1_gap_best_threshold": train_metrics_best["f1"] - val_metrics_best["f1"]
    }


print("=" * 80)
print("Starting permutation importance feature selection")

perm_sample_size = min(20000, X_val_ready.shape[0])
n_repeats = 3
top_feature_counts = [100, 150, 200]

print("Best previous run:", best_run_name)
print("Best previous validation AUC:", best_val_auc)
print("Permutation sample size:", perm_sample_size)
print("Permutation repeats:", n_repeats)
print("Top feature counts to test:", top_feature_counts)

if best_use_smote:
    print("Applying SMOTE for base permutation model fit...")
    X_perm_fit, y_perm_fit = SMOTE(random_state=seed, k_neighbors=5).fit_resample(X_train_ready, y_train)
else:
    X_perm_fit, y_perm_fit = X_train_ready, y_train

base_perm_model = make_model(best_settings)

start = time.time()
base_perm_model.fit(X_perm_fit, y_perm_fit)
base_perm_fit_seconds = time.time() - start

base_metrics = calculate_auc_metrics_for_selected_model(
    base_perm_model,
    X_train_ready,
    y_train,
    X_val_ready,
    y_val
)

print("Base permutation model validation AUC:", round(base_metrics["val_auc"], 6))
print("Base permutation model fit seconds:", round(base_perm_fit_seconds, 2))

rng = np.random.RandomState(seed)

perm_indices = rng.choice(
    X_val_ready.shape[0],
    size=perm_sample_size,
    replace=False
)

X_val_perm = X_val_ready[perm_indices]
y_val_perm = y_val.iloc[perm_indices] if hasattr(y_val, "iloc") else y_val[perm_indices]

with mlflow.start_run(run_name="xgboost_permutation_feature_selection"):
    mlflow.log_param("base_run_name", best_run_name)
    mlflow.log_param("feature_selection_method", "permutation_importance")
    mlflow.log_param("permutation_scoring", "roc_auc")
    mlflow.log_param("permutation_sample_size", perm_sample_size)
    mlflow.log_param("permutation_n_repeats", n_repeats)
    mlflow.log_param("base_use_smote", best_use_smote)
    mlflow.log_metric("base_validation_auc", base_metrics["val_auc"])
    mlflow.log_metric("base_train_auc", base_metrics["train_auc"])
    mlflow.log_metric("base_auc_gap", base_metrics["auc_gap"])
    mlflow.log_metric("base_perm_model_fit_seconds", base_perm_fit_seconds)

    for key, value in best_settings.items():
        mlflow.log_param(f"base_{key}", value)

    print("Calculating permutation importance...")
    perm_start = time.time()

    perm_result = permutation_importance(
        base_perm_model,
        X_val_perm,
        y_val_perm,
        scoring="roc_auc",
        n_repeats=n_repeats,
        random_state=seed,
        n_jobs=-1
    )

    perm_seconds = time.time() - perm_start

    permutation_importance_df = pd.DataFrame({
        "feature_index": np.arange(X_val_ready.shape[1]),
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std
    }).sort_values("importance_mean", ascending=False)

    permutation_importance_df["rank"] = np.arange(1, len(permutation_importance_df) + 1)

    permutation_importance_df.to_csv("xgboost_permutation_importance.csv", index=False)

    mlflow.log_metric("permutation_seconds", perm_seconds)
    mlflow.log_artifact("xgboost_permutation_importance.csv")

    print("Permutation importance seconds:", round(perm_seconds, 2))
    print("Top 20 permutation-important features:")
    display(permutation_importance_df.head(20))

    selection_results = []
    best_perm_val_auc = -1
    best_perm_top_n = None
    best_perm_indices = None
    best_perm_model = None
    best_perm_metrics = None

    for top_n in top_feature_counts:
        print("=" * 80)
        print("Testing permutation-selected top features:", top_n)

        selected_indices = (
            permutation_importance_df
            .head(top_n)["feature_index"]
            .sort_values()
            .values
        )

        X_train_selected = X_train_ready[:, selected_indices]
        X_val_selected = X_val_ready[:, selected_indices]

        if best_use_smote:
            X_selected_fit, y_selected_fit = SMOTE(random_state=seed, k_neighbors=5).fit_resample(
                X_train_selected,
                y_train
            )
        else:
            X_selected_fit, y_selected_fit = X_train_selected, y_train

        selected_model = make_model(best_settings)

        selected_start = time.time()
        selected_model.fit(X_selected_fit, y_selected_fit)
        selected_fit_seconds = time.time() - selected_start

        selected_metrics = calculate_auc_metrics_for_selected_model(
            selected_model,
            X_train_selected,
            y_train,
            X_val_selected,
            y_val
        )

        run_name = f"xgb_permutation_top_{top_n}_features"

        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.log_param("feature_selection_method", "permutation_importance")
            mlflow.log_param("selected_feature_count", top_n)
            mlflow.log_param("base_run_name", best_run_name)
            mlflow.log_param("use_smote", best_use_smote)

            for key, value in best_settings.items():
                mlflow.log_param(key, value)

            mlflow.log_metric("fit_seconds", selected_fit_seconds)
            mlflow.log_metric("train_auc", selected_metrics["train_auc"])
            mlflow.log_metric("val_auc", selected_metrics["val_auc"])
            mlflow.log_metric("auc_gap", selected_metrics["auc_gap"])
            mlflow.log_metric("best_f1_threshold", selected_metrics["best_threshold"])
            mlflow.log_metric("best_threshold_f1", selected_metrics["best_threshold_f1"])
            mlflow.log_metric("average_precision_gap", selected_metrics["average_precision_gap"])
            mlflow.log_metric("f1_gap_best_threshold", selected_metrics["f1_gap_best_threshold"])

            log_metrics("train_t05", selected_metrics["train_metrics_05"])
            log_metrics("val_t05", selected_metrics["val_metrics_05"])
            log_metrics("train_best_threshold", selected_metrics["train_metrics_best"])
            log_metrics("val_best_threshold", selected_metrics["val_metrics_best"])

        selection_results.append({
            "run_name": run_name,
            "selected_feature_count": top_n,
            "fit_seconds": selected_fit_seconds,

            "train_auc": selected_metrics["train_auc"],
            "val_auc": selected_metrics["val_auc"],
            "auc_gap": selected_metrics["auc_gap"],

            "train_average_precision": selected_metrics["train_metrics_05"]["average_precision"],
            "val_average_precision": selected_metrics["val_metrics_05"]["average_precision"],
            "average_precision_gap": selected_metrics["average_precision_gap"],

            "val_accuracy_t05": selected_metrics["val_metrics_05"]["accuracy"],
            "val_balanced_accuracy_t05": selected_metrics["val_metrics_05"]["balanced_accuracy"],
            "val_precision_t05": selected_metrics["val_metrics_05"]["precision"],
            "val_recall_t05": selected_metrics["val_metrics_05"]["recall"],
            "val_f1_t05": selected_metrics["val_metrics_05"]["f1"],

            "best_f1_threshold": selected_metrics["best_threshold"],
            "best_threshold_f1": selected_metrics["best_threshold_f1"],
            "val_precision_best_threshold": selected_metrics["val_metrics_best"]["precision"],
            "val_recall_best_threshold": selected_metrics["val_metrics_best"]["recall"],
            "val_f1_best_threshold": selected_metrics["val_metrics_best"]["f1"],

            **best_settings
        })

        print("selected_feature_count:", top_n)
        print("train_auc:", round(selected_metrics["train_auc"], 6))
        print("val_auc:", round(selected_metrics["val_auc"], 6))
        print("auc_gap:", round(selected_metrics["auc_gap"], 6))
        print("fit seconds:", round(selected_fit_seconds, 2))
        print("val_average_precision:", round(selected_metrics["val_metrics_05"]["average_precision"], 6))
        print("val_precision_t05:", round(selected_metrics["val_metrics_05"]["precision"], 6))
        print("val_recall_t05:", round(selected_metrics["val_metrics_05"]["recall"], 6))
        print("val_f1_t05:", round(selected_metrics["val_metrics_05"]["f1"], 6))
        print("best_f1_threshold:", round(selected_metrics["best_threshold"], 6))
        print("val_f1_best_threshold:", round(selected_metrics["val_metrics_best"]["f1"], 6))

        if selected_metrics["val_auc"] > best_perm_val_auc:
            best_perm_val_auc = selected_metrics["val_auc"]
            best_perm_top_n = top_n
            best_perm_indices = selected_indices
            best_perm_model = selected_model
            best_perm_metrics = selected_metrics

        gc.collect()

    permutation_selection_results_df = pd.DataFrame(selection_results).sort_values(
        "val_auc",
        ascending=False
    )

    permutation_selection_results_df.to_csv("xgboost_permutation_selection_results.csv", index=False)
    mlflow.log_artifact("xgboost_permutation_selection_results.csv")

    print("=" * 80)
    print("Permutation feature selection results:")
    display(permutation_selection_results_df)

    print("Best permutation-selected feature count:", best_perm_top_n)
    print("Best permutation-selected validation AUC:", best_perm_val_auc)
    print("Original best validation AUC:", best_val_auc)

    mlflow.log_metric("best_permutation_val_auc", best_perm_val_auc)
    mlflow.log_param("best_permutation_top_n", best_perm_top_n)

    if best_perm_val_auc > best_val_auc:
        print("=" * 80)
        print("Permutation feature selection improved validation AUC.")
        print("Fitting final raw-data pipeline with permutation column selector...")

        final_perm_start = time.time()

        best_permutation_pipeline = make_full_pipeline_with_column_selector(
            settings=best_settings,
            use_smote=best_use_smote,
            selected_indices=best_perm_indices
        )

        best_permutation_pipeline.fit(X_full, y_full)

        final_perm_fit_seconds = time.time() - final_perm_start

        perm_test_pred = best_permutation_pipeline.predict_proba(X_test)[:, 1]
        perm_submission = sample_submission.copy()
        perm_submission["isFraud"] = perm_test_pred
        perm_submission.to_csv("submission_xgboost_permutation_selected.csv", index=False)

        with mlflow.start_run(run_name="best_xgboost_permutation_selected_model", nested=True):
            mlflow.log_param("base_run_name", best_run_name)
            mlflow.log_param("feature_selection_method", "permutation_importance")
            mlflow.log_param("selected_feature_count", best_perm_top_n)
            mlflow.log_param("use_smote", best_use_smote)
            mlflow.log_param("pipeline", "full raw-data inference pipeline with permutation-selected processed columns")

            for key, value in best_settings.items():
                mlflow.log_param(key, value)

            mlflow.log_metric("best_validation_auc", best_perm_val_auc)
            mlflow.log_metric("previous_best_validation_auc", best_val_auc)
            mlflow.log_metric("validation_auc_improvement", best_perm_val_auc - best_val_auc)
            mlflow.log_metric("final_fit_seconds", final_perm_fit_seconds)

            mlflow.log_artifact("submission_xgboost_permutation_selected.csv")

            with open("model_permutation_selected.pkl", "wb") as f:
                pickle.dump(best_permutation_pipeline, f)

            mlflow.log_artifact("model_permutation_selected.pkl")

            mlflow.sklearn.log_model(
                sk_model=best_permutation_pipeline,
                artifact_path="model",
                registered_model_name="model_ieee_fraud_xgboost_permutation_selected"
            )

        print("Saved submission_xgboost_permutation_selected.csv")
    else:
        print("=" * 80)
        print("Permutation feature selection did not improve validation AUC.")
        print("Keeping the original best XGBoost pipeline as the final selected model.")

Starting permutation importance feature selection
Best previous run: XGB_trees_600_depth_8_smote_False
Best previous validation AUC: 0.9174214753882118
Permutation sample size: 20000
Permutation repeats: 3
Top feature counts to test: [100, 150, 200]
Base permutation model validation AUC: 0.917421
Base permutation model fit seconds: 84.38
Calculating permutation importance...
Permutation importance seconds: 142.93
Top 20 permutation-important features:


,feature_index,importance_mean,importance_std,rank
10,10,0.012998,0.000895,1
1,1,0.012257,0.000380,2
3,3,0.009983,0.000973,3
2,2,0.009936,0.000933,4
22,22,0.007884,0.001977,5
193,193,0.006887,0.000628,6
20,20,0.006798,0.000304,7
11,11,0.006668,0.000583,8
6,6,0.005505,0.001033,9
38,38,0.005358,0.000788,10


Testing permutation-selected top features: 100
🏃 View run xgb_permutation_top_100_features at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/9aa9dac948664a728cbc7b1e39388e04
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
selected_feature_count: 100
train_auc: 0.985022
val_auc: 0.91722
auc_gap: 0.067802
fit seconds: 48.05
val_average_precision: 0.535533
val_precision_t05: 0.290639
val_recall_t05: 0.684547
val_f1_t05: 0.408038
best_f1_threshold: 0.811366
val_f1_best_threshold: 0.517451
Testing permutation-selected top features: 150
🏃 View run xgb_permutation_top_150_features at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/de35ffd1119540ad97a02b6879dd1030
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
selected_feature_count: 150
train_auc: 0.986522
val_auc: 0.91954
auc_gap: 0.066982
fit seconds: 57.23
val_average_precision: 0.53039
val_precision_t05: 0.

,run_name,selected_feature_count,fit_seconds,train_auc,val_auc,auc_gap,train_average_precision,val_average_precision,average_precision_gap,val_accuracy_t05,val_balanced_accuracy_t05,val_precision_t05,val_recall_t05,val_f1_t05,best_f1_threshold,best_threshold_f1,val_precision_best_threshold,val_recall_best_threshold,val_f1_best_threshold,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha
1,xgb_permutation_top_150_features,150,57.232383,0.986522,0.919540,0.066982,0.843442,0.530390,0.313052,0.930199,0.811513,0.285421,0.684055,0.402782,0.781260,0.510142,0.553032,0.473425,0.510142,600,0.035,8,20,0.7,0.6,10.0,1.0
2,xgb_permutation_top_200_features,200,74.864047,0.986152,0.918049,0.068102,0.842918,0.532569,0.310349,0.928421,0.815813,0.281331,0.694882,0.400511,0.781498,0.512652,0.548695,0.481053,0.512652,600,0.035,8,20,0.7,0.6,10.0,1.0
0,xgb_permutation_top_100_features,100,48.047657,0.985022,0.917220,0.067802,0.830061,0.535533,0.294528,0.931656,0.812504,0.290639,0.684547,0.408038,0.811366,0.517451,0.607700,0.450541,0.517451,600,0.035,8,20,0.7,0.6,10.0,1.0


Best permutation-selected feature count: 150
Best permutation-selected validation AUC: 0.9195400036579334
Original best validation AUC: 0.9174214753882118
Permutation feature selection improved validation AUC.
Fitting final raw-data pipeline with permutation column selector...
🏃 View run xgboost_permutation_feature_selection at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/0f5eb1b6e1d94ca6adf2ebbb70c70c3a
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3


IndexError: index 211 is out of bounds for axis 1 with size 211

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrozenFeaturePipe(BaseEstimator, TransformerMixin):
    def __init__(self, fitted_pipe):
        self.fitted_pipe = fitted_pipe

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.fitted_pipe.transform(X)


def make_frozen_permutation_pipeline(settings, use_smote, selected_indices, fitted_feature_pipe):
    steps = [
        ("fitted_feature_pipe", FrozenFeaturePipe(fitted_pipe=fitted_feature_pipe)),
        ("permutation_column_selector", ColumnIndexSelector(selected_indices=selected_indices)),
    ]

    if use_smote:
        steps.append(("smote", SMOTE(random_state=seed, k_neighbors=5)))

    steps.append(("model", make_model(settings)))

    return ImbPipeline(steps)


print("=" * 80)
print("Recovering final permutation-selected model registration")

print("X_train_ready feature count:", X_train_ready.shape[1])
print("Selected feature count:", len(best_perm_indices))
print("Max selected index:", max(best_perm_indices))
print("Best permutation validation AUC:", best_perm_val_auc)
print("Original best validation AUC:", best_val_auc)

if max(best_perm_indices) >= X_train_ready.shape[1]:
    raise ValueError("Selected feature index is outside X_train_ready feature count.")

if best_perm_val_auc > best_val_auc:
    print("Permutation feature selection improved validation AUC.")
    print("Fitting frozen-preprocessing raw-data pipeline...")

    final_perm_start = time.time()

    best_permutation_pipeline = make_frozen_permutation_pipeline(
        settings=best_settings,
        use_smote=best_use_smote,
        selected_indices=best_perm_indices,
        fitted_feature_pipe=feature_pipe
    )

    best_permutation_pipeline.fit(X_full, y_full)

    final_perm_fit_seconds = time.time() - final_perm_start

    perm_test_pred = best_permutation_pipeline.predict_proba(X_test)[:, 1]

    perm_submission = sample_submission.copy()
    perm_submission["isFraud"] = perm_test_pred
    perm_submission.to_csv("submission_xgboost_permutation_selected.csv", index=False)

    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name="best_xgboost_permutation_selected_model"):
        mlflow.log_param("base_run_name", best_run_name)
        mlflow.log_param("feature_selection_method", "permutation_importance")
        mlflow.log_param("selected_feature_count", best_perm_top_n)
        mlflow.log_param("use_smote", best_use_smote)
        mlflow.log_param("pipeline", "FrozenFeaturePipe -> ColumnIndexSelector -> optional SMOTE -> XGBoost")

        for key, value in best_settings.items():
            mlflow.log_param(key, value)

        mlflow.log_metric("best_validation_auc", best_perm_val_auc)
        mlflow.log_metric("previous_best_validation_auc", best_val_auc)
        mlflow.log_metric("validation_auc_improvement", best_perm_val_auc - best_val_auc)
        mlflow.log_metric("final_fit_seconds", final_perm_fit_seconds)

        mlflow.log_artifact("submission_xgboost_permutation_selected.csv")

        with open("model_permutation_selected.pkl", "wb") as f:
            pickle.dump(best_permutation_pipeline, f)

        mlflow.log_artifact("model_permutation_selected.pkl")

        mlflow.sklearn.log_model(
            sk_model=best_permutation_pipeline,
            artifact_path="model",
            registered_model_name="model_ieee_fraud_xgboost_permutation_selected"
        )

    print("Saved submission_xgboost_permutation_selected.csv")
    print("Registered model_ieee_fraud_xgboost_permutation_selected")

else:
    print("Permutation feature selection did not improve validation AUC.")
    print("Keep the original best XGBoost model.")

Recovering final permutation-selected model registration
X_train_ready feature count: 220
Selected feature count: 150
Max selected index: 218
Best permutation validation AUC: 0.9195400036579334
Original best validation AUC: 0.9174214753882118
Permutation feature selection improved validation AUC.
Fitting frozen-preprocessing raw-data pipeline...


2026/05/04 20:00:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:01:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'model_ieee_fraud_xgboost_permutation_selected'.
2026/05/04 20:01:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_ieee_fraud_xgboost_permutation_selected, version 1
Created version '1' of model 'model_ieee_fraud_xgboost_permutation_selected'.


🏃 View run best_xgboost_permutation_selected_model at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3/runs/a41c9b08a0c944a1a0f6300b8ac439ae
🧪 View experiment at: https://dagshub.com/lmars23/ml_assignment_2.mlflow/#/experiments/3
Saved submission_xgboost_permutation_selected.csv
Registered model_ieee_fraud_xgboost_permutation_selected
